# 🇮🇳 Unemployment in India – Analysis & ML Model

## 📋 What is this Notebook About?
This notebook explores **Unemployment data across Indian states** and builds a
Machine Learning model to **predict the Unemployment Rate**.

### Steps we will follow:
1. **Load & Understand** the data  
2. **Clean** missing values  
3. **Explore** the data with charts (EDA)  
4. **Prepare features** for Machine Learning  
5. **Train & Evaluate** a Random Forest model  
6. **Conclusion**


## 1️⃣ Import Libraries

In [ ]:
# Basic libraries every Data Scientist uses
import pandas as pd          # for working with tables (DataFrames)
import numpy as np           # for math operations

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
print("✅ All libraries imported successfully!")


## 2️⃣ Load the Dataset

In [ ]:
# Load the CSV file into a DataFrame
df = pd.read_csv("Unemployment_in_India.csv")

# Strip extra spaces from column names (common issue in real datasets)
df.columns = df.columns.str.strip()

print(f"✅ Dataset loaded!  Rows: {df.shape[0]}  |  Columns: {df.shape[1]}")
df.head()


## 3️⃣ Understand the Data
Let's check **column names, data types, and missing values**.


In [ ]:
print("🔍 Dataset Info:")
print(df.info())
print()
print("📊 Basic Statistics:")
df.describe().round(2)


In [ ]:
# How many missing values are in each column?
print("❓ Missing Values per Column:")
print(df.isnull().sum())
print(f"\nTotal missing rows: {df.isnull().any(axis=1).sum()}")


## 4️⃣ Data Cleaning
We drop rows with missing values because they are only a small fraction (~3.6%).


In [ ]:
df_clean = df.dropna().reset_index(drop=True)

# Parse the Date column properly
df_clean['Date'] = pd.to_datetime(df_clean['Date'], dayfirst=True)

# Extract useful time features
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Year']  = df_clean['Date'].dt.year

print(f"✅ Clean dataset: {df_clean.shape[0]} rows  |  {df_clean.shape[1]} columns")
df_clean.head()


## 5️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of Unemployment Rate
plt.figure(figsize=(10, 4))
sns.histplot(df_clean['Estimated Unemployment Rate (%)'], bins=30,
             kde=True, color='steelblue')
plt.title('Distribution of Unemployment Rate (%)', fontsize=14)
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Urban vs Rural Unemployment
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_clean, x='Area', y='Estimated Unemployment Rate (%)',
            palette=['#2196F3', '#4CAF50'])
plt.title('Urban vs Rural Unemployment Rate', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Top 10 states by average unemployment rate
top_states = (df_clean.groupby('Region')['Estimated Unemployment Rate (%)']
              .mean()
              .sort_values(ascending=False)
              .head(10))

plt.figure(figsize=(10, 5))
top_states.plot(kind='bar', color='tomato', edgecolor='black')
plt.title('Top 10 States by Average Unemployment Rate', fontsize=14)
plt.ylabel('Avg Unemployment Rate (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Unemployment trend over time
monthly = (df_clean.groupby(['Year', 'Month'])['Estimated Unemployment Rate (%)']
           .mean()
           .reset_index())
monthly['Date'] = pd.to_datetime(monthly[['Year','Month']].assign(day=1))

plt.figure(figsize=(12, 4))
plt.plot(monthly['Date'], monthly['Estimated Unemployment Rate (%)'],
         color='purple', linewidth=2)
plt.title('Monthly Average Unemployment Rate Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Unemployment Rate (%)')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
num_cols = ['Estimated Unemployment Rate (%)',
            'Estimated Employed',
            'Estimated Labour Participation Rate (%)',
            'Month', 'Year']

plt.figure(figsize=(8, 5))
sns.heatmap(df_clean[num_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()


## 6️⃣ Feature Engineering

In [ ]:
# Encode categorical columns into numbers
# (ML models only understand numbers, not text)
le_region = LabelEncoder()
le_area   = LabelEncoder()

df_ml = df_clean.copy()
df_ml['Region_enc'] = le_region.fit_transform(df_ml['Region'])
df_ml['Area_enc']   = le_area.fit_transform(df_ml['Area'])

# Select our input features (X) and target column (y)
FEATURES = ['Region_enc', 'Area_enc', 'Month', 'Year',
            'Estimated Employed',
            'Estimated Labour Participation Rate (%)']

TARGET = 'Estimated Unemployment Rate (%)'

X = df_ml[FEATURES]
y = df_ml[TARGET]

print(f"✅ Features shape : {X.shape}")
print(f"✅ Target shape   : {y.shape}")
print(f"\nFeature columns used:\n{FEATURES}")


## 7️⃣ Split Data into Train & Test Sets

In [ ]:
# 80% data for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")


## 8️⃣ Train Machine Learning Models
We train **two models** and compare them:
- **Random Forest** – an ensemble of many decision trees  
- **Gradient Boosting** – builds trees sequentially to fix errors


In [ ]:
# ── Random Forest ──
rf = RandomForestRegressor(n_estimators=200, max_depth=10,
                           random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred  = rf.predict(X_test)
rf_r2    = r2_score(y_test, rf_pred)
rf_mae   = mean_absolute_error(y_test, rf_pred)
rf_rmse  = np.sqrt(mean_squared_error(y_test, rf_pred))

print("🌳 Random Forest Results:")
print(f"   R² Score : {rf_r2:.4f}  (closer to 1 = better)")
print(f"   MAE      : {rf_mae:.4f}  (average error in %)")
print(f"   RMSE     : {rf_rmse:.4f}")


In [ ]:
# ── Gradient Boosting ──
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                max_depth=5, random_state=42)
gb.fit(X_train, y_train)

gb_pred  = gb.predict(X_test)
gb_r2    = r2_score(y_test, gb_pred)
gb_mae   = mean_absolute_error(y_test, gb_pred)
gb_rmse  = np.sqrt(mean_squared_error(y_test, gb_pred))

print("🚀 Gradient Boosting Results:")
print(f"   R² Score : {gb_r2:.4f}")
print(f"   MAE      : {gb_mae:.4f}")
print(f"   RMSE     : {gb_rmse:.4f}")


## 9️⃣ Compare Models

In [ ]:
results = pd.DataFrame({
    'Model'   : ['Random Forest', 'Gradient Boosting'],
    'R² Score': [rf_r2, gb_r2],
    'MAE'     : [rf_mae, gb_mae],
    'RMSE'    : [rf_rmse, gb_rmse]
})

print("📊 Model Comparison Table:")
print(results.to_string(index=False))

# Pick the best model
best_model = rf if rf_r2 >= gb_r2 else gb
best_name  = "Random Forest" if rf_r2 >= gb_r2 else "Gradient Boosting"
best_pred  = rf_pred if rf_r2 >= gb_r2 else gb_pred
print(f"\n🏆 Best Model: {best_name}  (R² = {max(rf_r2, gb_r2):.4f})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Actual vs Predicted ---
axes[0].scatter(y_test, best_pred, alpha=0.5, color='steelblue', edgecolor='k', s=40)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
axes[0].set_xlabel('Actual Unemployment Rate (%)')
axes[0].set_ylabel('Predicted Unemployment Rate (%)')
axes[0].set_title(f'{best_name}: Actual vs Predicted')
axes[0].legend()

# --- Feature Importance ---
feat_imp = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values()
feat_imp.plot(kind='barh', ax=axes[1], color='teal', edgecolor='black')
axes[1].set_title('Feature Importance')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()


In [ ]:
# Residual plot – shows where the model makes mistakes
residuals = y_test - best_pred

plt.figure(figsize=(10, 4))
plt.scatter(best_pred, residuals, alpha=0.5, color='darkorange', edgecolor='k', s=40)
plt.axhline(0, color='red', linestyle='--', lw=2)
plt.xlabel('Predicted Value')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Residual Plot (errors should be close to 0)')
plt.tight_layout()
plt.show()


## 🔟 Conclusion

### 📌 Key Findings from EDA
| Insight | Detail |
|---|---|
| **Urban > Rural** | Urban areas consistently show higher unemployment |
| **COVID Impact** | Sharp spike in unemployment visible around mid-2020 |
| **High-risk states** | Tripura, Haryana, Himachal Pradesh top the list |
| **Labour Participation** | Negatively correlated with unemployment rate |

---

### 🤖 Machine Learning Summary

| Model | R² Score | MAE | RMSE |
|---|---|---|---|
| Random Forest | See above | See above | See above |
| Gradient Boosting | See above | See above | See above |

- ✅ **Labour Participation Rate** and **Estimated Employed** are the strongest predictors.  
- ✅ **Region** and **Area (Urban/Rural)** also play a significant role.  
- ✅ The best model achieves a high R² score, meaning it explains most of the variance in unemployment rates.  

---

### 💡 What Can Be Done Next?
1. Add **external data** (GDP, population, sector growth) to improve the model  
2. Try **XGBoost or LightGBM** for potentially higher accuracy  
3. Build a **state-level forecasting model** using time-series methods  
4. Deploy the model as a simple **web app** using Streamlit  

> *"Data is the new oil — but only when refined into insight."*
